# 🚀 Employment Prediction Hackathon — R Testing Kit
This notebook walks you from raw data to your submission, step by step data
**The goal:** for each person in Round 9, predict the *probability* that they are **employed (1)** vs **unemployed (0)**.
Your submissions are scored on **AUC**, so always submit a probability between 0 and 1 — never just a hard 0/1.

## 🛠️ Step 1: Load tools and Data

Before we begin exploring and modelling the data, let’s load the required R packages and import the training and testing datasets. This gives us everything we need to start building our employment prediction model.



In [ ]:
# install.packages(c('scales','pROC'))  # uncomment on first run if needed
suppressMessages({
  library(dplyr)
  library(ggplot2)
  library(scales)
})

# Load the competition datasets (adjust the paths to match your Kaggle input folder)
data_dir <- '../round_8'
train_data <- read.csv(file.path(data_dir, 'rounds_1_7.csv'), stringsAsFactors = FALSE)
test_data  <- read.csv(file.path(data_dir, 'round_8.csv'),  stringsAsFactors = FALSE)

cat('Training rows:', nrow(train_data), '| columns:', ncol(train_data), '\n')
cat('Testing rows :', nrow(test_data),  '| columns:', ncol(test_data),  '\n')

# The target column in train_data is `employed_status` (1 = employed, 0 = unemployed).
# It is NOT present in test_data — that is what you predict.


Training rows: 6301 | columns: 33 
Testing rows : 4885 | columns: 33 


## Step 2: Statistical Testing & Feature Selection
Before modelling, look at what the data actually contains. We use different tests to determine what we can find out about the data and how variables interact with dependable variable.
> **Sizing tip:** on Kaggle/Jupyter, control chart size with `options(repr.plot.width = ..., repr.plot.height = ...)`
> *before* the plot — not with fixed panel sizes. We set sensible sizes per chart below so nothing gets cropped.


In [2]:
# A tidy plotting frame with readable labels
plot_df <- train_data %>%
  filter(!is.na(employed_status)) %>%
  mutate(Status = factor(employed_status, levels = c(0, 1),
                         labels = c('Unemployed', 'Employed')))

pal <- c('Unemployed' = '#E15759', 'Employed' = '#59A14F')

# A reusable, clean theme
theme_hack <- theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = 'bold'),
        plot.subtitle = element_text(color = 'grey35'),
        legend.position = 'top')


### 2.1 — ANOVA Table
Test whether the mean of a numerical variable differs significantly between employed and unemployed groups.

In [3]:
# ============================================
# 2.1 ANOVA Table
# ============================================

# Create employment groups for the ANOVA
train_data$employed_status_factor <- factor(
  train_data$employed_status,
  levels = c(0, 1),
  labels = c("Unemployed", "Employed")
)

# Variables to test
numeric_vars <- c(
  "age",
  "days_since_last_obs",
  "total_historical_rounds",
  "tenure_lag",
  "work_readiness_score"
)

# Keep only variables that exist in the dataset
numeric_vars <- numeric_vars[
  numeric_vars %in% names(train_data)
]

# Run ANOVA for each numerical variable
for (variable in numeric_vars) {

  formula <- as.formula(
    paste(variable, "~ employed_status_factor")
  )

  anova_model <- aov(
    formula,
    data = train_data
  )

  cat("\n========================================\n")
  cat("ANOVA Test:", variable, "\n")
  cat("========================================\n")

  print(summary(anova_model))
}


ANOVA Test: age 
                         Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor    1    698   698.3   37.37 1.04e-09 ***
Residuals              6233 116486    18.7                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
66 observations deleted due to missingness

ANOVA Test: days_since_last_obs 
                         Df   Sum Sq Mean Sq F value Pr(>F)  
employed_status_factor    1    39024   39024   5.009 0.0253 *
Residuals              2359 18377940    7791                 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
3940 observations deleted due to missingness

ANOVA Test: total_historical_rounds 
                         Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor    1   20.1  20.089   43.53 4.52e-11 ***
Residuals              6236 2878.0   0.462                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
63 observations deleted due to missingness

ANOVA T

**What it means:** the classes are imbalanced (~1 in 3 employed). A model that predicts 'unemployed' for everyone would look ~68% 'accurate' but is useless — this is exactly why the competition uses **AUC**, which rewards ranking people by their probability of employment rather than raw accuracy.

## Generic Anova table 

In [4]:
# ============================================
# 2.1 ANOVA Table
# ============================================

# Create employment groups for the ANOVA
train_data$employed_status_factor <- factor(
  train_data$employed_status,
  levels = c(0, 1),
  labels = c("Unemployed", "Employed")
)

# Variables to test
numeric_vars <- c(
  "age",
  "days_since_last_obs",
  "total_historical_rounds",
  "tenure_lag",
  "work_readiness_score"
)

# Keep only variables that exist in the dataset
numeric_vars <- numeric_vars[
  numeric_vars %in% names(train_data)
]

# Run ANOVA for each numerical variable
for (variable in numeric_vars) {

  formula <- as.formula(
    paste(variable, "~ employed_status_factor")
  )

  anova_model <- aov(
    formula,
    data = train_data
  )

  cat("\n========================================\n")
  cat("ANOVA Test:", variable, "\n")
  cat("========================================\n")

  print(summary(anova_model))
}


ANOVA Test: age 
                         Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor    1    698   698.3   37.37 1.04e-09 ***
Residuals              6233 116486    18.7                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
66 observations deleted due to missingness

ANOVA Test: days_since_last_obs 
                         Df   Sum Sq Mean Sq F value Pr(>F)  
employed_status_factor    1    39024   39024   5.009 0.0253 *
Residuals              2359 18377940    7791                 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
3940 observations deleted due to missingness

ANOVA Test: total_historical_rounds 
                         Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor    1   20.1  20.089   43.53 4.52e-11 ***
Residuals              6236 2878.0   0.462                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
63 observations deleted due to missingness

ANOVA T

### 2.2 — Stepwise Regression
Use stepwise selection to identify variables that contribute most strongly to the model.

In [5]:
# ============================================
# 2.2 Stepwise Logistic Regression
# ============================================

# Candidate predictor variables
candidate_vars <- c(
  "age",
  "gender",
  "race",
  "sa_citizen",
  "province",
  "education_level",
  "education_schooling_grade_twelve_equivalent",
  "institution_type",
  "school_quintile",
  "education_field",
  "employed_lag",
  "status_lag",
  "status_broad_lag",
  "tenure_lag",
  "total_historical_rounds",
  "days_since_last_obs",
  "work_readiness_score"
)

# Keep only columns that exist
candidate_vars <- candidate_vars[
  candidate_vars %in% names(train_data)
]

# --------------------------------------------
# Create modelling dataset
# --------------------------------------------

model_vars <- c(
  "employed_status",
  candidate_vars
)

stepwise_data <- train_data[, model_vars, drop = FALSE]

# Remove rows with missing values
stepwise_data <- na.omit(stepwise_data)

cat("Rows available for modelling:",
    nrow(stepwise_data), "\n")

# --------------------------------------------
# Remove predictors with only one level
# AFTER removing missing values
# --------------------------------------------

valid_vars <- c()

cat("\nChecking predictor variables:\n")

for (variable in candidate_vars) {

  # Number of unique values remaining
  # after NA rows have been removed
  number_of_levels <- length(
    unique(stepwise_data[[variable]])
  )

  if (number_of_levels >= 2) {

    valid_vars <- c(valid_vars, variable)

    cat("✓", variable,
        "- ", number_of_levels, " levels\n")

  } else {

    cat("✗", variable,
        "- ONLY ", number_of_levels,
        " level; removed\n")
  }
}

# --------------------------------------------
# Check target variable
# --------------------------------------------

target_levels <- unique(stepwise_data$employed_status)

cat("\nEmployment status levels:\n")
print(target_levels)

if (length(target_levels) < 2) {

  stop(
    "Stepwise regression cannot run because ",
    "employed_status has fewer than 2 levels."
  )
}

# --------------------------------------------
# Convert categorical variables to factors
# --------------------------------------------

categorical_vars <- c(
  "gender",
  "race",
  "sa_citizen",
  "province",
  "education_level",
  "education_schooling_grade_twelve_equivalent",
  "institution_type",
  "education_field",
  "employed_lag",
  "status_lag",
  "status_broad_lag"
)

categorical_vars <- intersect(
  categorical_vars,
  valid_vars
)

for (variable in categorical_vars) {
  stepwise_data[[variable]] <- factor(
    stepwise_data[[variable]]
  )
}

# --------------------------------------------
# Build logistic regression formula
# --------------------------------------------

full_formula <- as.formula(
  paste(
    "employed_status ~",
    paste(valid_vars, collapse = " + ")
  )
)

cat("\n========================================\n")
cat("Full Logistic Regression Model\n")
cat("========================================\n")

print(full_formula)

# --------------------------------------------
# Fit full model
# --------------------------------------------

full_model <- glm(
  full_formula,
  data = stepwise_data,
  family = binomial()
)

summary(full_model)

# --------------------------------------------
# Stepwise selection
# --------------------------------------------

cat("\n========================================\n")
cat("Stepwise Regression\n")
cat("========================================\n")

stepwise_model <- step(
  full_model,
  direction = "both",
  trace = TRUE
)

# --------------------------------------------
# Final model
# --------------------------------------------

cat("\n========================================\n")
cat("Final Stepwise Model\n")
cat("========================================\n")

summary(stepwise_model)

# --------------------------------------------
# Selected variables
# --------------------------------------------

cat("\n========================================\n")
cat("Selected Variables\n")
cat("========================================\n")

print(names(coef(stepwise_model)))

Rows available for modelling: 444 

Checking predictor variables:
✓ age -  19  levels
✓ gender -  2  levels
✓ race -  2  levels
✗ sa_citizen - ONLY  1  level; removed
✓ province -  9  levels
✓ education_level -  4  levels
✗ institution_type - ONLY  1  level; removed
✓ school_quintile -  6  levels
✗ education_field - ONLY  1  level; removed
✓ employed_lag -  2  levels
✓ status_lag -  4  levels
✓ status_broad_lag -  4  levels
✓ tenure_lag -  208  levels
✓ total_historical_rounds -  3  levels
✓ days_since_last_obs -  106  levels
✓ work_readiness_score -  361  levels



Employment status levels:
[1] 1 0

Full Logistic Regression Model
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + employed_lag + status_lag + status_broad_lag + 
    tenure_lag + total_historical_rounds + days_since_last_obs + 
    work_readiness_score



Call:
glm(formula = full_formula, family = binomial(), data = stepwise_data)

Coefficients: (4 not defined because of singularities)
                                         Estimate Std. Error z value Pr(>|z|)
(Intercept)                             6.137e-01  1.617e+00   0.379  0.70438
age                                     1.610e-02  3.702e-02   0.435  0.66358
genderMale                              5.949e-01  2.393e-01   2.486  0.01292
raceColoured                            1.498e+00  5.737e-01   2.612  0.00900
provinceFree State                     -1.388e+00  6.832e-01  -2.031  0.04221
provinceGauteng                        -4.647e-01  4.949e-01  -0.939  0.34777
provinceKwaZulu-Natal                  -8.123e-01  5.472e-01  -1.484  0.13773
provinceLimpopo                         9.074e-02  5.949e-01   0.153  0.87877
provinceMpumalanga                      4.916e-01  5.837e-01   0.842  0.39963
provinceNorth West                     -2.694e-01  5.972e-01  -0.451  0.65193
province


Stepwise Regression
Start:  AIC=503.16
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + employed_lag + status_lag + status_broad_lag + 
    tenure_lag + total_historical_rounds + days_since_last_obs + 
    work_readiness_score


Step:  AIC=503.16
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + employed_lag + status_lag + tenure_lag + 
    total_historical_rounds + days_since_last_obs + work_readiness_score


Step:  AIC=503.16
employed_status ~ age + gender + race + province + education_level + 
    school_quintile + status_lag + tenure_lag + total_historical_rounds + 
    days_since_last_obs + work_readiness_score

                          Df Deviance    AIC
- total_historical_rounds  1   457.21 501.21
- tenure_lag               1   457.28 501.28
- age                      1   457.35 501.35
<none>                         457.16 503.16
- province                 8   473.31 503.31
- education_level  


Call:
glm(formula = employed_status ~ gender + race + province + education_level + 
    school_quintile + status_lag + days_since_last_obs + work_readiness_score, 
    family = binomial(), data = stepwise_data)

Coefficients:
                                     Estimate Std. Error z value Pr(>|z|)   
(Intercept)                          1.662167   1.163538   1.429  0.15313   
genderMale                           0.604827   0.238013   2.541  0.01105 * 
raceColoured                         1.494638   0.573924   2.604  0.00921 **
provinceFree State                  -1.380562   0.677258  -2.038  0.04150 * 
provinceGauteng                     -0.455987   0.495794  -0.920  0.35772   
provinceKwaZulu-Natal               -0.811316   0.546432  -1.485  0.13761   
provinceLimpopo                      0.124611   0.591598   0.211  0.83317   
provinceMpumalanga                   0.521014   0.580593   0.897  0.36951   
provinceNorth West                  -0.252276   0.596325  -0.423  0.67226   
pro


Selected Variables
 [1] "(Intercept)"                         "genderMale"                         
 [3] "raceColoured"                        "provinceFree State"                 
 [5] "provinceGauteng"                     "provinceKwaZulu-Natal"              
 [7] "provinceLimpopo"                     "provinceMpumalanga"                 
 [9] "provinceNorth West"                  "provinceNorthern Cape"              
[11] "provinceWestern Cape"                "education_levelGrade 11/NQF Level 3"
[13] "education_levelGrade 12/NQF Level 4" "education_levelGrade 9/NQF Level 1" 
[15] "school_quintile"                     "status_lagunemployed"               
[17] "status_lagwage and self employed"    "status_lagwage employed"            
[19] "days_since_last_obs"                 "work_readiness_score"               


In [6]:
# Variables selected by stepwise regression

selected_variables <- names(
  coef(stepwise_model)
)

selected_variables

[1] "(Intercept)"                         "genderMale"                         
 [3] "raceColoured"                        "provinceFree State"                 
 [5] "provinceGauteng"                     "provinceKwaZulu-Natal"              
 [7] "provinceLimpopo"                     "provinceMpumalanga"                 
 [9] "provinceNorth West"                  "provinceNorthern Cape"              
[11] "provinceWestern Cape"                "education_levelGrade 11/NQF Level 3"
[13] "education_levelGrade 12/NQF Level 4" "education_levelGrade 9/NQF Level 1" 
[15] "school_quintile"                     "status_lagunemployed"               
[17] "status_lagwage and self employed"    "status_lagwage employed"            
[19] "days_since_last_obs"                 "work_readiness_score"

**What it means:** the lag columns are powerful *when present* but empty for most people. Decide deliberately how to handle those NAs — dropping them throws away most of your data. A common trick (used in Step 5) is to treat 'missing history' as its own category rather than discarding the row.

## 📋 2.3 Item Reliability Analysis (IRA)

Item Reliability Analysis evaluates whether multiple items intended to measure the same underlying concept are consistent with one another. In this case, we are interested in whether the available work-readiness assessment items provide a reliable measure of work readiness.

We will use **Cronbach's alpha** to measure internal consistency. A higher alpha generally indicates that the items are measuring a similar underlying construct.

**Your task:** Examine the reliability output and identify whether the items appear to form a reliable scale that could be useful for employment prediction.


In [7]:
# ============================================
# 2.3 Item Reliability Analysis (IRA)
# ============================================

# Install if needed:
# install.packages("psych")

library(psych)

# ------------------------------------------------
# Find potential work-readiness item variables
# ------------------------------------------------

# Look for columns containing "work_readiness"
work_readiness_items <- names(train_data)[
  grepl("work_readiness", names(train_data), ignore.case = TRUE)
]

cat("Potential work-readiness variables:\n")
print(work_readiness_items)


# ------------------------------------------------
# Remove the overall score if it exists
# ------------------------------------------------

work_readiness_items <- work_readiness_items[
  work_readiness_items != "work_readiness_score"
]

cat("\nItems used for reliability analysis:\n")
print(work_readiness_items)


# ------------------------------------------------
# Check whether enough individual items exist
# ------------------------------------------------

if (length(work_readiness_items) >= 2) {
  
  # Select the item columns
  reliability_data <- train_data[
    , work_readiness_items,
    drop = FALSE
  ]
  
  # Convert columns to numeric where possible
  reliability_data <- data.frame(
    lapply(reliability_data, function(x) {
      as.numeric(as.character(x))
    })
  )
  
  # Run Cronbach's Alpha
  reliability_result <- psych::alpha(
    reliability_data,
    check.keys = TRUE
  )
  
  # Display the complete reliability output
  print(reliability_result)
  
  # Display Cronbach's alpha specifically
  cat("\n----------------------------------------\n")
  cat("Cronbach's Alpha:", reliability_result$total$raw_alpha, "\n")
  cat("----------------------------------------\n")
  
  # Display item-level statistics
  cat("\nItem Statistics:\n")
  print(reliability_result$item.stats)
  
  # Display alpha if each item is removed
  cat("\nAlpha if Item is Deleted:\n")
  print(reliability_result$alpha.drop)
  
} else {
  
  cat("\nNot enough individual work-readiness items were found.\n")
  cat("At least two items are required for Item Reliability Analysis.\n")
  cat("The dataset currently appears to contain only the overall\n")
  cat("work_readiness_score rather than individual assessment items.\n")
}


Attaching package: 'psych'


The following objects are masked from 'package:scales':

    alpha, rescale


The following objects are masked from 'package:ggplot2':

    %+%, alpha




Potential work-readiness variables:
[1] "work_readiness_score"

Items used for reliability analysis:
character(0)

Not enough individual work-readiness items were found.
At least two items are required for Item Reliability Analysis.
The dataset currently appears to contain only the overall
work_readiness_score rather than individual assessment items.


**What it means:** someone employed last round is more likely to be employed now than someone who was unemployed. This 'stickiness' is part of the story — but as you'll see, it is not the whole picture, and turning it into leaderboard gains takes care.

### 2.4 — Chi-Square Test
Test whether categorical variables are significantly associated with employed_status.

In [8]:
# 2.4 Chi-Square Tests
# Test whether categorical variables are associated with employment status

categorical_vars <- c(
  "gender",
  "race",
  "sa_citizen",
  "province",
  "education_level",
  "education_schooling_grade_twelve_equiv",
  "institution_type",
  "education_field",
  "status_lag",
  "status_broad_lag"
)

# Keep only variables that exist
categorical_vars <- categorical_vars[
  categorical_vars %in% names(train_data)
]

for (variable in categorical_vars) {
  
  # Create contingency table
  contingency_table <- table(
    train_data[[variable]],
    train_data$employed_status
  )
  
  # Run chi-square test
  chi_test <- chisq.test(contingency_table)
  
  cat("\n========================================\n")
  cat("Chi-Square Test:", variable, "\n")
  cat("========================================\n")
  
  print(contingency_table)
  print(chi_test)
}


Chi-Square Test: gender 
        
            0    1
  Female 2497  896
  Male   1778 1067

	Pearson's Chi-squared test with Yates' continuity correction

data:  contingency_table
X-squared = 87.85, df = 1, p-value < 2.2e-16



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: race 
               
                   0    1
                   6    2
  African/Black 3991 1794
  Asian            3    0
  Coloured       239  147
  Indian          27   12
  Other            6    0
  Unspecified      0    1
  White            3    7

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 21.751, df = 7, p-value = 0.002805



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: sa_citizen 
       
           0    1
  FALSE    2    4
  TRUE  4273 1959

	Pearson's Chi-squared test with Yates' continuity correction

data:  contingency_table
X-squared = 2.0099, df = 1, p-value = 0.1563



...


Chi-Square Test: institution_type 
                                      
                                          0    1
                                       3648 1688
  Comprehensive University              128   66
  Private College                        80   36
  Private Higher Education Institution   48   18
  Public College                        209   80
  Public University                      72   31
  Tech Accelerator                        1    1
  Trade School                           13    4
  University of Technology               76   39

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 4.3483, df = 8, p-value = 0.8244



Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: education_field 
                                                         
                                                             0    1
                                                          2898 1213
  Accounting and bookkeeping                                40   17
  Administration                                           105   46
  Advertising                                                2    1
  Agriculture                                               18    7
  Aquaculture and fisheries                                  1    0
  Architecture                                               1    0
  Arts and crafts                                            5    2
  Auditing                                                   5    3
  Automotive and transport equipment manufacturing           2    0
  Aviation and maritime transport                            0    1
  Beauty and wellness                                        2    0
  Biological sciences a

Warning message in chisq.test(contingency_table):
"Chi-squared approximation may be incorrect"



Chi-Square Test: status_broad_lag 
                        
                            0    1
                         2760 1140
  self employed           101   90
  studying                138   31
  unclear                   2    1
  unemployed             1054  407
  wage and self employed   18   29
  wage employed           202  265

	Pearson's Chi-squared test

data:  contingency_table
X-squared = 211.37, df = 6, p-value < 2.2e-16



## 🧪 2.5 Statistical Test Template

Use this template to investigate another relationship in the dataset.

Your test should:

1. State what relationship you are investigating.
2. Identify the variables involved.
3. Run an appropriate statistical test.
4. Record the important statistic and p-value.
5. Decide whether the result is statistically significant.
6. Explain what the result means for employment prediction.

In [9]:
# ============================================
# 2.5 Statistical Test Template
# ============================================

# Variable being investigated
variable_1 <- "REPLACE_WITH_VARIABLE"

# Target variable
variable_2 <- "employed_status"

# Your statistical test here
# Example:
#
# test_result <- chisq.test(
#   table(train_data[[variable_1]], train_data[[variable_2]])
# )
#
# print(test_result)


# Record the important result
# statistic <- ...
# p_value <- ...


# Your conclusion
cat("Variable tested:", variable_1, "\n")
cat("Target variable:", variable_2, "\n")
cat("Test statistic: [SUBMIT VALUE]\n")
cat("P-value: [SUBMIT VALUE]\n")

Variable tested: REPLACE_WITH_VARIABLE 
Target variable: employed_status 
Test statistic: [SUBMIT VALUE]
P-value: [SUBMIT VALUE]


**What it means:** if the boxes overlap heavily, the score alone is a weak separator — and adding weak features to a model can even *hurt* it. Part of the challenge is finding which combinations genuinely carry signal.

2.6 ANOVCA Table

In [10]:
# ============================================
# 2.6 ANOVCA Table
# ============================================

# Create employment groups for the ANOVA
train_data$employed_status_factor <- factor(
  train_data$employed_status,
  levels = c(0, 1),
  labels = c("Unemployed", "Employed")
)

# Variables to test
numeric_vars <- c(
  "age",
  "days_since_last_obs",
  "total_historical_rounds",
  "tenure_lag",
  "work_readiness_score"
)

# Keep only variables that exist in the dataset
numeric_vars <- numeric_vars[
  numeric_vars %in% names(train_data)
]

# Run ANOVA for each numerical variable
for (variable in numeric_vars) {

  formula <- as.formula(
    paste(variable, "~ employed_status_factor")
  )

  anova_model <- aov(
    formula,
    data = train_data
  )

  cat("\n========================================\n")
  cat("ANOVA Test:", variable, "\n")
  cat("========================================\n")

  print(summary(anova_model))
}


ANOVA Test: age 
                         Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor    1    698   698.3   37.37 1.04e-09 ***
Residuals              6233 116486    18.7                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
66 observations deleted due to missingness

ANOVA Test: days_since_last_obs 
                         Df   Sum Sq Mean Sq F value Pr(>F)  
employed_status_factor    1    39024   39024   5.009 0.0253 *
Residuals              2359 18377940    7791                 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
3940 observations deleted due to missingness

ANOVA Test: total_historical_rounds 
                         Df Sum Sq Mean Sq F value   Pr(>F)    
employed_status_factor    1   20.1  20.089   43.53 4.52e-11 ***
Residuals              6236 2878.0   0.462                     
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
63 observations deleted due to missingness

ANOVA T

## Round and Observation Variables

The initial analysis suggests that `lag_round`, `days_since_last_obs`, and `current_round` do not show a statistically significant relationship with employment status. However, these variables should be tested further before being completely excluded from the predictive model.

A particularly strong relationship was observed for `employed_lag`. The very large F-statistic and extremely small p-value indicate a strong association between previous employment status and current employment status. This supports the earlier observation that **someone who was employed previously has a high likelihood of being employed in the future**.

## `employed_lag` and Missing History

`employed_lag` contains a substantial amount of missing data:

* Missing: **11,451 / 16,719 observations**
* Approximately **68.5%** of the values are missing.

This missingness is important because a missing value does not necessarily mean that the participant was unemployed. Instead, it may indicate that **there is no previous observation or employment history available**.

Therefore, rather than treating missing values as ordinary missing data, we can separate the information into two variables:

* **`employed_lag`** — the previous employment status when historical information exists.
* **`employed_lag_missing`** — a binary indicator showing whether previous employment information is missing.

For example:

```text
No previous history
    ↓
employed_lag_missing = 1
employed_lag = NA

Previous history exists
    ↓
employed_lag_missing = 0
employed_lag = 0 or 1
```

This allows the model to distinguish between **previously unemployed**, **previously employed**, and **having no previous employment history**, rather than incorrectly treating missing history as unemployment.


In [11]:
    train_data$employed_lag_missing <- ifelse(
    is.na(train_data$employed_lag), 1, 0)

    
    test_data$employed_lag_missing <- ifelse(
    is.na(test_data$employed_lag), 1, 0)


  

## `employed_lag_missing`

`employed_lag_missing` is a new binary variable that indicates whether the `employed_lag` value is missing.

* **`1`** → `employed_lag` is missing
* **`0`** → `employed_lag` is available

This can be useful for modelling because the **absence of previous employment information may itself contain predictive information**. The model can therefore distinguish between participants with a recorded employment history and those without one.


## `work_readiness_score` and `wr_filled`

`work_readiness_score` and `wr_filled` provide related information about work readiness. Using both directly as predictors may introduce redundancy.

Instead, we can use `wr_filled` as a **binary indicator** showing whether a work-readiness score is available, while `work_readiness_score` contains the actual score when it is present.

This allows us to distinguish between:

* **`wr_filled = 1`** → work-readiness score is available
* **`wr_filled = 0`** → work-readiness score is missing

The missingness itself may contain useful information for predicting employment, so we can test it separately rather than treating the two variables as identical predictors.


In [12]:
train_data$wr_missing <- ifelse(
  is.na(train_data$work_readiness_score), 1, 0
)

test_data$wr_missing <- ifelse(
  is.na(test_data$work_readiness_score), 1, 0
)

## Investigation of the catergorical variables:

status_lag
status_broad_lag
education_level
education_field
institution_type
gender
province
race



In [13]:
# Chi-squared test for categorical variables
chisq.test(
  table(
    train_data$status_lag,
    train_data$employed_status
  )
)

# Chi-squared test for status_broad_lag vs employed_status
chisq.test(
  table(
    train_data$status_broad_lag,
    train_data$employed_status
  )
)

# Chi-squared test for education_level vs employed_status
chisq.test(
  table(
    train_data$education_level,
    train_data$employed_status
  )
)


	Pearson's Chi-squared test

data:  table(train_data$status_lag, train_data$employed_status)
X-squared = 203.61, df = 4, p-value < 2.2e-16


Warning message in chisq.test(table(train_data$status_broad_lag, train_data$employed_status)):
"Chi-squared approximation may be incorrect"



	Pearson's Chi-squared test

data:  table(train_data$status_broad_lag, train_data$employed_status)
X-squared = 211.37, df = 6, p-value < 2.2e-16


Warning message in chisq.test(table(train_data$education_level, train_data$employed_status)):
"Chi-squared approximation may be incorrect"



	Pearson's Chi-squared test

data:  table(train_data$education_level, train_data$employed_status)
X-squared = 44.186, df = 29, p-value = 0.03524


## Conclusion of the chi-squared test 
| Variable | χ² | df | p-value | Result |
|---|---:|---:|---:|---|
| status_lag | 485.05 | 5 | < 2.2e-16 | Extremely strong |
| status_broad_lag | 471.29 | 6 | < 2.2e-16 | Extremely strong |
| education_level | 44.37 | 35 | 0.1333 | Not significant |


**education_level**
We don't have enough evidence to say education level is associated with employment status in this dataset.

However, I wouldn't immediately remove it. As the p-value is not too high above 0.1 so it still play a factor as logically education still a deciding factor especially NQF levels.

## 🏗️ Step 3: A first baseline model
We predict a binary outcome, so we start with **logistic regression**. For this first attempt we deliberately
This sets a (low) floor — and shows that simply throwing variables at the model is not enough.

In [14]:
# Fill missing work readiness with the training median so every row has a value
wr_median <- median(train_data$work_readiness_score, na.rm = TRUE)
train_data$wr_filled <- ifelse(is.na(train_data$work_readiness_score), wr_median, train_data$work_readiness_score)
test_data$wr_filled  <- ifelse(is.na(test_data$work_readiness_score),  wr_median, test_data$work_readiness_score)

# Treat citizenship as a category
train_data$sa_citizen_chr <- as.character(train_data$sa_citizen)
test_data$sa_citizen_chr  <- as.character(test_data$sa_citizen)

baseline_model <- glm(employed_status ~ wr_filled + sa_citizen_chr,
                      data = train_data, family = binomial)

summary(baseline_model)



Call:
glm(formula = employed_status ~ wr_filled + sa_citizen_chr, family = binomial, 
    data = train_data)

Coefficients:
                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)          0.2141     0.8710   0.246   0.8058    
wr_filled            1.0094     0.1950   5.177 2.25e-07 ***
sa_citizen_chrTRUE  -1.4749     0.8665  -1.702   0.0887 .  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 7770  on 6237  degrees of freedom
Residual deviance: 7740  on 6235  degrees of freedom
  (63 observations deleted due to missingness)
AIC: 7746

Number of Fisher Scoring iterations: 4


Description of all the variables

In [15]:
variable_check <- data.frame(
  variable = names(train_data),
  type = sapply(train_data, function(x) class(x)[1]),
  unique_values = sapply(train_data, function(x) length(unique(x))),
  missing = sapply(train_data, function(x) sum(is.na(x))),
  missing_percent = sapply(train_data, function(x) mean(is.na(x)) * 100)
)

variable_check

,variable,type,unique_values,missing,missing_percent
,<chr>,<chr>,<int>,<int>,<dbl>
anonymised_id,anonymised_id,character,6301,0,0.00000000
lag_round,lag_round,integer,5,3933,62.41866370
current_round,current_round,integer,5,0,0.00000000
days_since_last_obs,days_since_last_obs,integer,144,3933,62.41866370
total_historical_rounds,total_historical_rounds,integer,4,0,0.00000000
survey_date,survey_date,character,108,0,0.00000000
sample,sample,character,14,0,0.00000000
sample_first,sample_first,character,13,0,0.00000000
employed_status,employed_status,integer,3,63,0.99984130


## 🔮 Step 4: Predict and build a submission
For AUC we submit **probabilities**, not hard labels. The block below writes a valid `submission_baseline.csv`.
Upload it on the competition's Submit page to see your public leaderboard score.

In [16]:
# Make sure the test set has the same prepared column the model was trained on
test_data$sa_citizen_chr <- as.character(test_data$sa_citizen)

# Predicted PROBABILITY of employment (0-1)
test_prob <- predict(baseline_model, newdata = test_data, type = 'response')

submission <- data.frame(anonymised_id = test_data$anonymised_id,
                         employed_status = test_prob)
write.csv(submission, 'submission_baseline.csv', row.names = FALSE, quote = FALSE)
cat('Saved submission_baseline.csv with', nrow(submission), 'rows\n')
head(submission)


Saved submission_baseline.csv with 4885 rows


,anonymised_id,employed_status
,<chr>,<dbl>
1,ID_10003,0.3193994
2,ID_10006,0.3121229
3,ID_10015,0.4352314
4,ID_10018,0.3053840
5,ID_10023,0.3121229
6,ID_10027,0.2668010


## 🔄 Step 5: Choose better features and watch the score improve
The baseline features barely helped. Now we swap in features that actually carry signal:
`gender` and `education_level`, while keeping `work_readiness_score`. Same model, better inputs — and the score improves.
This is the core lesson: **which** variables you choose matters more than how many.

In [17]:
# Fill missing work readiness (recompute to be safe)
wr_median <- median(train_data$work_readiness_score, na.rm = TRUE)
train_data$wr_filled <- ifelse(is.na(train_data$work_readiness_score), wr_median, train_data$work_readiness_score)
test_data$wr_filled  <- ifelse(is.na(test_data$work_readiness_score),  wr_median, test_data$work_readiness_score)

# Align education_level factor levels between train and test
train_data$education_level <- factor(train_data$education_level)
test_data$education_level  <- factor(test_data$education_level, levels = levels(train_data$education_level))

upgraded_model <- glm(employed_status ~ gender + education_level + wr_filled,
                      data = train_data, family = binomial)

up_prob <- predict(upgraded_model, newdata = test_data, type = 'response')
up_prob[is.na(up_prob)] <- mean(train_data$employed_status, na.rm = TRUE)

submission2 <- data.frame(anonymised_id = test_data$anonymised_id, employed_status = up_prob)
write.csv(submission2, 'submission_upgraded.csv', row.names = FALSE, quote = FALSE)
cat('Saved submission_upgraded.csv\n')
head(submission2)


Saved submission_upgraded.csv


,anonymised_id,employed_status
,<chr>,<dbl>
1,ID_10003,0.2858276
2,ID_10006,0.3215119
3,ID_10015,0.3826617
4,ID_10018,0.2742249
5,ID_10023,0.2739121
6,ID_10027,0.3542180


# 🧪 Historical Round Evaluator

The competition uses AUC to evaluate predicted employment probabilities.

Because the outcomes for Rounds 6–8 are known, we can simulate the competition by training on earlier rounds and evaluating predictions on a future round.

This allows us to test whether a model generalises to future survey rounds before submitting predictions for Round 9.

In [20]:
# ============================================
# Local Kaggle-Style Evaluator
# ============================================
# install.packages("pROC")  # Uncomment if not already installed
library(pROC)

evaluate_submission <- function(
    actual,
    predicted,
    model_name = "Model"
) {

  # Check lengths
  if (length(actual) != length(predicted)) {
    stop("Actual and predicted values have different lengths.")
  }

  # Check for invalid predictions
  if (any(!is.finite(predicted))) {
    stop("Predictions contain NA, NaN, or infinite values.")
  }

  # Check probability range
  if (any(predicted < 0 | predicted > 1)) {
    stop("Predictions must be between 0 and 1.")
  }

  # Remove missing actual outcomes
  valid <- !is.na(actual)

  actual_clean <- actual[valid]
  predicted_clean <- predicted[valid]

  # Calculate ROC
  roc_result <- roc(
    actual_clean,
    predicted_clean,
    quiet = TRUE
  )

  # Calculate AUC
  auc_score <- as.numeric(
    auc(roc_result)
  )

  # Display result
  cat("\n============================================\n")
  cat("Model:", model_name, "\n")
  cat("============================================\n")

  cat(
    "Observations evaluated:",
    length(actual_clean),
    "\n"
  )

  cat(
    "AUC:",
    round(auc_score, 5),
    "\n"
  )

  cat(
    "Percentage:",
    round(auc_score * 100, 2),
    "%\n"
  )

  cat("============================================\n")

  return(auc_score)
}

## Run your model normally

### 3.1 Model

In [21]:
# ============================================
# Historical Test — Logistic Regression 3.1
# Rounds 1-5 → Round 6
# ============================================

# --------------------------------------------
# 1. Calculate training median
# --------------------------------------------

wr_median_31 <- median(
  train_data$work_readiness_score,
  na.rm = TRUE
)

cat(
  "Training work-readiness median:",
  wr_median_31,
  "\n"
)


# --------------------------------------------
# 2. Fill missing work-readiness values
# --------------------------------------------

train_data$work_readiness_score <- ifelse(
  is.na(train_data$work_readiness_score),
  wr_median_31,
  train_data$work_readiness_score
)

test_data$work_readiness_score <- ifelse(
  is.na(test_data$work_readiness_score),
  wr_median_31,
  test_data$work_readiness_score
)


# --------------------------------------------
# 3. Make citizenship a factor
# --------------------------------------------

train_data$sa_citizen <- factor(
  train_data$sa_citizen
)

test_data$sa_citizen <- factor(
  test_data$sa_citizen,
  levels = levels(train_data$sa_citizen)
)


# --------------------------------------------
# 4. Fit logistic regression
# --------------------------------------------

model_31 <- glm(
  employed_status ~
    work_readiness_score +
    sa_citizen,
  data = train_data,
  family = binomial()
)


# --------------------------------------------
# 5. Predict Round 6
# --------------------------------------------

pred_31 <- predict(
  model_31,
  newdata = test_data,
  type = "response"
)


# --------------------------------------------
# 6. Check predictions
# --------------------------------------------

cat(
  "Number of predictions:",
  length(pred_31),
  "\n"
)

cat(
  "NA predictions:",
  sum(is.na(pred_31)),
  "\n"
)

cat(
  "Invalid predictions:",
  sum(!is.finite(pred_31)),
  "\n"
)


# --------------------------------------------
# 7. Evaluate AUC
# --------------------------------------------

round_6_score_31 <- evaluate_submission(
  actual = test_data$employed_status,
  predicted = pred_31,
  model_name = "3.1 Logistic Regression — Rounds 1-7 → Round 8"
)

Training work-readiness median: 0.466248 
Number of predictions: 4885 
NA predictions: 0 
Invalid predictions: 0 

Model: 3.1 Logistic Regression — Rounds 1-7 → Round 8 
Observations evaluated: 4883 
AUC: 0.55366 
Percentage: 55.37 %


### 3.1 Historical Benchmark

The logistic regression using `work_readiness_score` and `sa_citizen`
achieved an AUC of **0.55366 (55.37%)** when trained on Rounds 1–5
and evaluated on Round 6.

This is only moderately above the random baseline of 0.50, suggesting
that this simple model has limited ability to generalise to a future
survey round.

This score will serve as the historical benchmark for subsequent
logistic regression experiments.

## 🚀 Over to you — ideas to climb the leaderboard
You just saw that swapping weak features for better ones lifted the score. There is a lot more room to improve:

- **Use participant history:** `employed_lag`, `status_lag`, `status_broad_lag`, `tenure_lag` — a person's *previous* status is a strong clue, when it is present.
- **Handle missingness thoughtfully:** treat 'missing' as its own category; train and test differ in how complete some columns are.
- **More demographics & geography:** `age`, `race`, `province`, `district`, `municipality`.
- **Education & skills:** `school_quintile`, the `matric_*` subjects.
- **Try stronger models:** `ranger` (random forest) or `xgboost`.
- **Always submit probabilities** and validate with AUC, not accuracy.

Good luck!